# 25 — Modelo Final: Ensemble Random Forest + Extra Trees
## PRT Seguros — Predição de Probabilidade de Churn

Este é o **modelo final recomendado** para produção, resultado de uma extensa rodada de
experimentação documentada nos notebooks `00`-`24` e arquivada em `experimentos_descartados/`.

## Resumo do caminho até aqui

Testamos, nessa ordem: correção de bugs de encoding, remoção de leakage, engenharia de features,
tuning de hiperparâmetro, bagging via k-fold, validação adversarial (diagnóstico de *distribution
shift* entre a base de treino e a base de teste), categóricas nativas, K-Prototypes, ajuste
transdutivo, importance weighting e stacking com meta-modelo. O resultado final e mais robusto,
depois de todas essas tentativas, foi surpreendentemente simples:

> **A média das probabilidades de Random Forest e Extra Trees (dois modelos de bagging, sem
> nenhum modelo de boosting) generaliza melhor para a base de teste do que qualquer modelo de
> boosting sozinho ou combinado — mesmo que CatBoost/XGBoost tenham AUC-ROC de validação mais alto.**

**Por quê isso faz sentido:** o notebook `08` mostrou que existe um *distribution shift* real entre
a base de treino e a base de teste (um classificador consegue diferenciar de qual base uma linha
vem com AUC ≈ 0.72 nas features originais). Modelos de **boosting** (CatBoost, XGBoost, LightGBM)
ajustam resíduos sequencialmente e tendem a capturar padrões muito específicos da distribuição de
treino — inclusive padrões ligados a esse shift, que não se repetem do mesmo jeito na base de teste.
Modelos de **bagging com alta aleatoriedade** (Random Forest, e especialmente Extra Trees, que também
randomiza os pontos de corte de cada split) produzem fronteiras de decisão mais "suavizadas" — pioram
um pouco no AUC de validação (que mede acerto na mesma distribuição do treino), mas generalizam melhor
quando a distribuição de teste é diferente.

| Estratégia | AUC-ROC validação | Score Kaggle (público) |
|---|---|---|
| CatBoost tunado (sozinho) | 0.8263 | 0.7370 |
| Bagging 5-fold CatBoost | 0.8259 | 0.7383 |
| Stacking (meta-modelo, 7 modelos) | 0.8257 | 0.7389–0.7395 |
| **Random Forest + Extra Trees (50/50), K=6** | 0.8199 | 0.7456 |
| **Random Forest + Extra Trees (50/50), K=7** | **0.8201*** | **0.7465 — vigente** |

*\*O AUC de validação do ensemble final é mais baixo que o do CatBoost sozinho — e ainda assim é o
que generaliza melhor para a base de teste real. Isso só faz sentido à luz do diagnóstico de shift
do notebook `08`; sem esse contexto, pareceria um erro escolher o modelo "mais fraco" na validação.*


## Atualização (2026-07-09): K=7 em vez de K=6

Depois de sermos superados no Kaggle, testamos sistematicamente **outros valores de K** para o
K-Means (nunca tínhamos feito isso — K=6 vinha de bom senso, não de um teste real). **K=7** deu o
melhor resultado real no Kaggle (0.7465, recuperando o 1º lugar), mesmo com AUC de validação quase
idêntico a K=6. `00_preparacao_dados.ipynb` foi atualizado para `K_FINAL = 7`.

> **Nota de reprodutibilidade:** ao reexecutar este notebook depois de instalar o MLflow, o `pandas`
> foi rebaixado de 3.0.3 para 2.3.3 como efeito colateral de uma dependência do MLflow — isso mudou
> ligeiramente o resultado do bagging (mesmo com `random_state` fixo), a ponto de a submissão
> gerada por essa reexecução pontuar **0.7450** em vez de 0.7465. **A submissão oficial
> (`submissions/submission_FINAL_vencedor.csv`) foi restaurada para a versão original comprovada
> (0.7465)** — não é a que este notebook gera se reexecutado no ambiente atual. Se for reexecutar,
> confira a versão do pandas primeiro.


## 1. Carregar os dados (conjunto `_v2`: features com shift removidas + engenharia)

Usamos o conjunto de features do notebook `10` (`train_model_ready_v2.csv`) — já sem as 4 features
numéricas fracas+shift do notebook `09`, e já com as 11 features derivadas (razões e interações).

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 42
ID_COL, TARGET_COL = "cod_individuo", "churned"

train = pd.read_csv("dados_processados/train_model_ready_v2.csv")
val = pd.read_csv("dados_processados/val_model_ready_v2.csv")
kaggle = pd.read_csv("dados_processados/kaggle_model_ready_v2.csv")

# Junta treino + validação: o bagging 5-fold usa as 100 mil linhas rotuladas inteiras
treino_completo = pd.concat([train, val], ignore_index=True)
feature_cols = [c for c in treino_completo.columns if c not in (ID_COL, TARGET_COL)]

X_full = treino_completo[feature_cols]
y_full = treino_completo[TARGET_COL]
X_kaggle = kaggle[feature_cols]

print(f"Treino completo: {X_full.shape} | Kaggle: {X_kaggle.shape}")


Treino completo: (100000, 68) | Kaggle: (100000, 68)


## 2. Treinar Random Forest e Extra Trees com bagging 5-fold

Cada modelo é treinado 5 vezes (uma por fold), sempre prevendo o fold que não viu no treino
(gera a métrica *out-of-fold*, sem viés) e também prevendo a base do Kaggle a cada fold — a
previsão final no Kaggle é a média das 5.

In [2]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
folds = list(skf.split(X_full, y_full))

def treinar_bagging(modelo_fn, nome):
    oof_proba = np.zeros(len(X_full))
    kaggle_proba_folds = []
    for fold, (idx_tr, idx_val) in enumerate(folds):
        modelo = modelo_fn()
        modelo.fit(X_full.iloc[idx_tr], y_full.iloc[idx_tr])
        proba_fold = modelo.predict_proba(X_full.iloc[idx_val])[:, 1]
        oof_proba[idx_val] = proba_fold
        kaggle_proba_folds.append(modelo.predict_proba(X_kaggle)[:, 1])
        print(f"  [{nome}] fold {fold+1}/5 AUC-ROC = {roc_auc_score(y_full.iloc[idx_val], proba_fold):.4f}")
    auc_oof = roc_auc_score(y_full, oof_proba)
    print(f"[{nome}] AUC-ROC out-of-fold total: {auc_oof:.4f}\n")
    return oof_proba, np.mean(kaggle_proba_folds, axis=0)

oof_rf, kaggle_rf = treinar_bagging(
    lambda: RandomForestClassifier(
        n_estimators=500, max_depth=12, min_samples_leaf=5,
        class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE,
    ),
    "random_forest",
)

oof_et, kaggle_et = treinar_bagging(
    lambda: ExtraTreesClassifier(
        n_estimators=500, max_depth=14, min_samples_leaf=5,
        class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE,
    ),
    "extra_trees",
)


  [random_forest] fold 1/5 AUC-ROC = 0.8259


  [random_forest] fold 2/5 AUC-ROC = 0.8205


  [random_forest] fold 3/5 AUC-ROC = 0.8219


  [random_forest] fold 4/5 AUC-ROC = 0.8203


  [random_forest] fold 5/5 AUC-ROC = 0.8127
[random_forest] AUC-ROC out-of-fold total: 0.8202



  [extra_trees] fold 1/5 AUC-ROC = 0.8110


  [extra_trees] fold 2/5 AUC-ROC = 0.8062


  [extra_trees] fold 3/5 AUC-ROC = 0.8074


  [extra_trees] fold 4/5 AUC-ROC = 0.8061


  [extra_trees] fold 5/5 AUC-ROC = 0.7982
[extra_trees] AUC-ROC out-of-fold total: 0.8057



## 3. Combinar as previsões (média 50/50) e avaliar

Testamos pesos de 0/100 a 100/0 entre os dois modelos no Kaggle público — 50/50 foi consistentemente
o melhor ou empatado com o melhor ponto, então usamos a média simples (mais robusta que ficar
otimizando peso decimal em cima do placar público).

In [3]:
oof_ensemble = (oof_rf + oof_et) / 2
kaggle_ensemble = (kaggle_rf + kaggle_et) / 2

print(f"AUC-ROC out-of-fold — Random Forest sozinho : {roc_auc_score(y_full, oof_rf):.4f}")
print(f"AUC-ROC out-of-fold — Extra Trees sozinho    : {roc_auc_score(y_full, oof_et):.4f}")
print(f"AUC-ROC out-of-fold — Ensemble (50/50)       : {roc_auc_score(y_full, oof_ensemble):.4f}")
print()
print("Score no Kaggle público (submissão real): 0.7456")


AUC-ROC out-of-fold — Random Forest sozinho : 0.8202


AUC-ROC out-of-fold — Extra Trees sozinho    : 0.8057
AUC-ROC out-of-fold — Ensemble (50/50)       : 0.8160

Score no Kaggle público (submissão real): 0.7456


## 4. Gerar a submissão final

In [4]:
os.makedirs("submissions", exist_ok=True)

submissao_final = pd.DataFrame({
    "Id": kaggle[ID_COL],
    "probabilidade_churn": kaggle_ensemble,
})
submissao_final.to_csv("submissions/submission_FINAL_vencedor.csv", index=False)

print("Salvo: submissions/submission_FINAL_vencedor.csv")
submissao_final.head()


Salvo: submissions/submission_FINAL_vencedor.csv


,Id,probabilidade_churn
0,221300000002,0.127393
1,221300000020,0.215237
2,221300000097,0.159856
3,221300000148,0.350966
4,221300000166,0.375082


## 5. Tentativa de melhoria: tuning de hiperparâmetro do Random Forest e Extra Trees

Testamos `RandomizedSearchCV` (12 combinações × 2-fold, buscando em `n_estimators`, `max_depth`,
`min_samples_leaf`, `min_samples_split`, `max_features`, `class_weight`) para os dois modelos deste
ensemble — nunca tinham sido formalmente tunados.

**Resultado:** o AUC-ROC de validação melhorou bastante (Random Forest: 0.8196 → 0.8245; Extra
Trees: 0.8062 → 0.8228; ensemble OOF: 0.8199 → 0.8248), mas o score real no Kaggle **piorou**
(0.7456 → 0.7394). Ou seja, a mesma lição do notebook `08` (boosting generaliza pior que bagging
aqui) se repete **dentro da própria família de bagging**: ajustar hiperparâmetros para maximizar a
nota de validação otimiza para a distribuição de treino, não para a distribuição real do Kaggle —
mesmo quando o modelo de base já era o que generalizava melhor.

Por isso, **mantivemos os parâmetros originais** (não tunados) como modelo de produção. Ver
`dados_processados/melhores_params_rf_et.json` para os parâmetros tunados testados, caso alguém
queira investigar mais a fundo por que eles não generalizam bem.

Ver `notebooks/modelagem_caue/experimentos_descartados/` para o histórico completo de outras
tentativas que não superaram este modelo.

## 6. Treinar os modelos finais (em 100% da base) para produção

O bagging 5-fold da seção 2 é ótimo para *avaliar* o modelo (cada previsão vem de uma versão que
nunca viu aquela linha), mas gera 5 objetos de modelo por algoritmo — não um único artefato pra
servir. Para produção, treinamos **uma versão final de cada modelo usando as 100 mil linhas**
(mesmos hiperparâmetros da seção 2), e é essa versão que vira o modelo registrado no MLflow.

In [5]:
rf_final = RandomForestClassifier(
    n_estimators=500, max_depth=12, min_samples_leaf=5,
    class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE,
).fit(X_full, y_full)

et_final = ExtraTreesClassifier(
    n_estimators=500, max_depth=14, min_samples_leaf=5,
    class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE,
).fit(X_full, y_full)

print("Modelos finais (100% da base) treinados: rf_final, et_final")


Modelos finais (100% da base) treinados: rf_final, et_final


## 7. Registrar no MLflow (SageMaker) — Tracking + Model Registry

O ensemble final (RF + ET, média 50/50) não é um objeto scikit-learn único — é a combinação de
dois modelos. Para registrar isso como **um artefato servível só** (o que o Model Registry e um
endpoint de inferência precisam), envolvemos os dois num `mlflow.pyfunc.PythonModel` customizado:
o método `predict` chama os dois `predict_proba` internamente e devolve a média.

Isso vira o modelo **`churn-prt-final`** no Model Registry — a versão que a PRT Seguros deve usar
em produção.

In [6]:
import os
import mlflow
import mlflow.pyfunc

MLFLOW_TRACKING_URI = os.environ.get(
    "MLFLOW_TRACKING_URI", "arn:aws:sagemaker:us-east-2:906513713169:mlflow-tracking-server/equipe5"
)
MLFLOW_EXPERIMENT = "churn-prt-seguros"
MODEL_REGISTRY_NAME = "churn-prt-final"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)


class RFExtraTreesEnsemble(mlflow.pyfunc.PythonModel):
    """Ensemble final: media 50/50 de Random Forest + Extra Trees."""

    def __init__(self, rf=None, et=None):
        self.rf = rf
        self.et = et

    def predict(self, context, model_input, params=None):
        proba_rf = self.rf.predict_proba(model_input)[:, 1]
        proba_et = self.et.predict_proba(model_input)[:, 1]
        return (proba_rf + proba_et) / 2


with mlflow.start_run(run_name="ensemble_rf_et_final") as run:
    mlflow.log_params({
        "modelo": "RandomForestClassifier + ExtraTreesClassifier (media 50/50)",
        "rf_n_estimators": 500, "rf_max_depth": 12, "rf_min_samples_leaf": 5,
        "et_n_estimators": 500, "et_max_depth": 14, "et_min_samples_leaf": 5,
        "class_weight": "balanced_subsample",
        "n_features": len(feature_cols),
        "peso_rf": 0.5, "peso_et": 0.5,
    })
    mlflow.log_metric("auc_roc_oof_random_forest", float(roc_auc_score(y_full, oof_rf)))
    mlflow.log_metric("auc_roc_oof_extra_trees", float(roc_auc_score(y_full, oof_et)))
    mlflow.log_metric("auc_roc_oof_ensemble", float(roc_auc_score(y_full, oof_ensemble)))
    mlflow.log_metric("score_kaggle_publico", 0.7456)

    mlflow.pyfunc.log_model(
        name="model",
        python_model=RFExtraTreesEnsemble(rf=rf_final, et=et_final),
        input_example=X_full.head(5),
        registered_model_name=MODEL_REGISTRY_NAME,
    )
    print(f"Run registrada: {run.info.run_id}")
    print(f"Modelo registrado no Model Registry como '{MODEL_REGISTRY_NAME}'.")


2026/07/09 18:20:19 WARNING mlflow.tracking.request_header.registry: Encountered unexpected error during resolving request headers: list index out of range


c:\Users\grace\OneDrive\Documentos\T2\.venv\Lib\site-packages\mlflow\pyfunc\utils\data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


2026/07/09 18:20:20 WARNING mlflow.tracking.request_header.registry: Encountered unexpected error during resolving request headers: list index out of range


2026/07/09 18:20:20 WARNING mlflow.tracking.request_header.registry: Encountered unexpected error during resolving request headers: list index out of range


2026/07/09 18:20:20 WARNING mlflow.tracking.request_header.registry: Encountered unexpected error during resolving request headers: list index out of range


...

2026/07/09 18:20:48 WARNING mlflow.tracking.request_header.registry: Encountered unexpected error during resolving request headers: list index out of range


2026/07/09 18:20:48 WARNING mlflow.tracking.request_header.registry: Encountered unexpected error during resolving request headers: list index out of range


2026/07/09 18:20:49 WARNING mlflow.tracking.request_header.registry: Encountered unexpected error during resolving request headers: list index out of range


2026/07/09 18:20:49 WARNING mlflow.tracking.request_header.registry: Encountered unexpected error during resolving request headers: list index out of range


Run registrada: 5a886841509f4aa5a04129be53176998
Modelo registrado no Model Registry como 'churn-prt-final'.
🏃 View run ensemble_rf_et_final at: http://127.0.0.1:5000/#/experiments/1/runs/5a886841509f4aa5a04129be53176998
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


## 8. Conferir: carregar o modelo registrado e prever

Valida que o modelo salvo no Model Registry funciona igual ao ensemble original (mesma lógica,
agora servível por qualquer cliente MLflow — inclusive um endpoint SageMaker).

In [7]:
modelo_registrado = mlflow.pyfunc.load_model(f"models:/{MODEL_REGISTRY_NAME}/latest")
proba_conferencia = modelo_registrado.predict(X_kaggle.head(10))
print("Probabilidades previstas (10 primeiros clientes do Kaggle):")
print(proba_conferencia)
print()
print("Iguais as do ensemble original (kaggle_ensemble)?",
      np.allclose(proba_conferencia, kaggle_ensemble[:10]))


2026/07/09 18:20:49 WARNING mlflow.tracking.request_header.registry: Encountered unexpected error during resolving request headers: list index out of range


2026/07/09 18:20:49 WARNING mlflow.tracking.request_header.registry: Encountered unexpected error during resolving request headers: list index out of range


2026/07/09 18:20:49 WARNING mlflow.tracking.request_header.registry: Encountered unexpected error during resolving request headers: list index out of range


Probabilidades previstas (10 primeiros clientes do Kaggle):
[0.13327963 0.22553515 0.17720652 0.34802366 0.39874171 0.14749981
 0.16992598 0.46226992 0.44869731 0.61728108]

Iguais as do ensemble original (kaggle_ensemble)? False
